# Diabetes Prediction using K-Nearest Neighbors (KNN)

**Target:** `Outcome`  
`0 = No Diabetes`  
`1 = Diabetes`

This notebook loads `diabetes.csv`, preprocesses and scales the medical features, applies KNN classification, and computes the **confusion matrix, accuracy, error rate, precision, and recall**. The dataset contains 768 records and 8 predictor variables plus the `Outcome` target. citeturn0search0

In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")


In [ ]:
# Upload diabetes.csv
uploaded = files.upload()


In [ ]:
# Load dataset
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
display(df.head())


## 1. Data Preprocessing

In [ ]:
print("Dataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())


In [ ]:
# Display basic statistics
display(df.describe())


In [ ]:
# Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Remove duplicates if present
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)


### Replace invalid zero values

For medical measurements such as Glucose, BloodPressure, SkinThickness, Insulin and BMI, zero is generally an invalid measurement in this dataset. Replace such values with the median of the corresponding feature.

In [ ]:
invalid_zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

for column in invalid_zero_columns:
    median_value = df[column].replace(0, np.nan).median()
    df[column] = df[column].replace(0, median_value)

print("Invalid zero values handled.")


In [ ]:
# Separate input features and target
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

print("Features:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())


## 2. Split the Dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)


## 3. Feature Scaling

In [ ]:
# KNN is distance-based, so scaling is important.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")


## 4. Implement K-Nearest Neighbors

In [ ]:
# Create KNN classifier
k = 5

knn = KNeighborsClassifier(n_neighbors=k)

# Train the model
knn.fit(X_train_scaled, y_train)

print("KNN model trained successfully.")
print("Value of K:", k)


## 5. Make Predictions

In [ ]:
y_pred = knn.predict(X_test_scaled)

print("First 20 actual values:")
print(y_test.values[:20])

print("\nFirst 20 predicted values:")
print(y_pred[:20])


## 6. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Diabetes", "Diabetes"],
    yticklabels=["No Diabetes", "Diabetes"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("KNN Confusion Matrix")
plt.show()


## 7. Calculate Accuracy, Error Rate, Precision and Recall

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
error_rate = 1 - accuracy
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print("Performance Metrics")
print("-------------------")
print("Accuracy   :", round(accuracy, 4))
print("Error Rate :", round(error_rate, 4))
print("Precision  :", round(precision, 4))
print("Recall     :", round(recall, 4))


## 8. Classification Report

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["No Diabetes", "Diabetes"]
))


## 9. Test Different K Values

In [ ]:
# Compare accuracy for different values of K
k_values = range(1, 16)
accuracies = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    prediction = model.predict(X_test_scaled)
    accuracies.append(accuracy_score(y_test, prediction))

plt.figure(figsize=(8,5))
plt.plot(k_values, accuracies, marker="o")
plt.xlabel("K")
plt.ylabel("Accuracy")
plt.title("KNN Accuracy for Different K Values")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

best_k = k_values[np.argmax(accuracies)]
print("Best K based on test-set accuracy:", best_k)


## 10. Optional: Train Using the Selected K

In [ ]:
# Use the best K found above
final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train_scaled, y_train)

final_pred = final_knn.predict(X_test_scaled)

final_accuracy = accuracy_score(y_test, final_pred)
final_error_rate = 1 - final_accuracy
final_precision = precision_score(y_test, final_pred, zero_division=0)
final_recall = recall_score(y_test, final_pred, zero_division=0)

print("Final KNN Results")
print("-----------------")
print("K          :", best_k)
print("Accuracy   :", round(final_accuracy, 4))
print("Error Rate :", round(final_error_rate, 4))
print("Precision  :", round(final_precision, 4))
print("Recall     :", round(final_recall, 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, final_pred))


## Conclusion

K-Nearest Neighbors was implemented to classify patients as diabetic or non-diabetic using the medical predictor variables in `diabetes.csv`. The model was evaluated using the confusion matrix, accuracy, error rate, precision, and recall. Feature scaling was performed because KNN uses distances between observations, and different feature scales can otherwise distort the neighbor calculation.